# tRNA-Arg-TCT-4-1 Dependency Map

Predict the secondary structure of tRNA-Arg-TCT-4-1 (chr1:159141611-159141684, minus strand)
using three different approaches, then compare each against the known cloverleaf contact map.

**Note:** This gene was previously labeled tRNA41/tRNA-Ala but is actually **tRNA-Arg** (anticodon TCT)
per [GtRNAdb](https://gtrnadb.ucsc.edu/genomes/eukaryota/Hsapi38/genes/tRNA-Arg-TCT-4-1.html).

**Sections:**
1. Setup & ground truth
2. **RiNALMo epistasis geometry** — double-mutant embeddings, `epi_R_singles` metric
3. **SpeciesLM log-odds dependency map** — MLM head, single-pass O(N)
4. **SpeciesLM embedding perturbation map** — cosine distance on token embeddings, single-pass O(N)
5. Summary comparison
6. **Beyond contacts** — high-scoring pairs NOT in the secondary structure
7. **3D distance matrix** — compare against PDB structure (captures stacking & tertiary contacts)

## 1. Setup & ground truth

In [ ]:
import sys
from pathlib import Path

# Ensure genebeddings is importable (editable install or sys.path fallback)
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations, product
from scipy.stats import pearsonr
from seqmat import SeqMat

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_logodds_dependency_map,
    compute_embedding_perturbation_map,
)

In [ ]:
# --- Model config (change these to switch models) ---
from genebeddings.wrappers import RiNALMoWrapper, SpeciesLMWrapper

# Epistasis metrics model
EPI_MODEL_CLS = RiNALMoWrapper
EPI_CONTEXT = 511
EPI_DB_PATH = 'embeddings/rinalmo.db'

# Dependency map model (needs MLM head for log-odds; any model works for embedding perturbation)
DEPMAP_MODEL_CLS = SpeciesLMWrapper
DEPMAP_CONTEXT = 600

# --- tRNA41 (tRNA-Ala) coordinates ---
CHROM = 'chr1'
START = 159141611
END = 159141684
GENE = 'tRNA41'
CHROM_NUM = '1'
STRAND = 'N'  # negative strand
MAX_DISTANCE = 100
BASES = 'ATGC'
PAD = 256  # flanking context for dependency maps

### Helpers

In [ ]:
BASES = 'ATGC'

def get_all_epistasis_ids(seq, indices, gene, chrom, zyg="N", max_distance=100):
    """Generate epistasis IDs for all position pairs within max_distance, all non-ref ALT combos."""
    items = []
    for i, j in combinations(range(len(seq)), 2):
        if j - i > max_distance:
            continue
        pos1, pos2 = int(indices[i]), int(indices[j])
        ref1, ref2 = seq[i], seq[j]
        if ref1 not in BASES or ref2 not in BASES:
            continue
        alts1 = [b for b in BASES if b != ref1]
        alts2 = [b for b in BASES if b != ref2]
        for alt1, alt2 in product(alts1, alts2):
            id1 = f"{gene}:{chrom}:{pos1}:{ref1}:{alt1}:{zyg}"
            id2 = f"{gene}:{chrom}:{pos2}:{ref2}:{alt2}:{zyg}"
            items.append({'epistasis_id': f"{id1}|{id2}", 'pos1': pos1, 'pos2': pos2})
    return pd.DataFrame(items)


def dotbracket_to_contact_map(ss):
    """Convert dot-bracket notation to a symmetric binary contact matrix."""
    opens = "([{<ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    closes = ")]}>abcdefghijklmnopqrstuvwxyz"
    N = len(ss)
    M = np.zeros((N, N), dtype=int)
    stack = {ch: [] for ch in opens}
    for i, ch in enumerate(ss):
        if ch in opens:
            stack[ch].append(i)
        elif ch in closes:
            j = stack[opens[closes.index(ch)]].pop()
            M[i, j] = M[j, i] = 1
    return M


def upper_tri_flatten(M):
    i, j = np.triu_indices_from(M, k=1)
    return M[i, j]


def zero_diagonal(M):
    """Zero out the diagonal of a matrix."""
    M = M.copy()
    np.fill_diagonal(M, 0)
    return M


def correlate_with_structure(pred_matrix, M_true, label, flip=True):
    """Score a predicted map against the true contact map.
    
    Zeros out the diagonal of both maps before comparing (upper triangle Pearson r).
    """
    pred = pred_matrix.copy()
    if flip:
        pred = pred[::-1, ::-1]
    pred[np.isnan(pred)] = 0
    pred = zero_diagonal(pred)
    truth = zero_diagonal(M_true)
    true_flat = upper_tri_flatten(truth)
    pred_flat = upper_tri_flatten(pred)
    r, p = pearsonr(true_flat, pred_flat)
    print(f"{label}: Pearson r = {r:.4f}, p = {p:.2e}")
    return r, p


def plot_heatmap(matrix, title, cmap='magma', figsize=(10, 8)):
    """Plot a symmetric heatmap."""
    plt.figure(figsize=figsize)
    sns.heatmap(matrix, cmap=cmap, robust=True)
    plt.title(title)
    plt.xlabel("Position")
    plt.ylabel("Position")
    plt.tight_layout()
    plt.show()

### Ground truth: tRNA secondary structure

In [ ]:
# tRNA41 cloverleaf (reverse-complement strand, gaps stripped from alignment)
ss_raw = '.>>>>>>>..>>>>..........<<<<.>>>>>.........................<<<<<................>>>>>.......<<<<<<<<<<<<.'
ss_raw = ss_raw.replace('>', '(').replace('<', ')')
refseq_raw = '.GTCTCTGTGGCGCAATGGAcgA.GCGCGCTGGACTTCTA..................ATCCAGAG...........GtTCCGGGTTCGAGTCCCGGCAGAGATG'
ss = ''.join(c for c, r in zip(ss_raw, refseq_raw) if r != '.')
M_true = dotbracket_to_contact_map(ss)

plt.figure(figsize=(6, 6))
plt.imshow(M_true, cmap='viridis', origin='upper')
plt.title("Ground truth: tRNA41 secondary structure")
plt.xlabel("Position")
plt.ylabel("Position")
plt.colorbar()
plt.show()
print(f"Structure: {len(ss)} nt, {M_true.sum() // 2} base pairs")

# Collect results for final comparison
all_results = []

## 2. RiNALMo: Epistasis geometry (double-mutant embeddings)

For every pair of positions (i, j), embed all 9 double-mutant combinations,
compute epistasis metrics (non-additivity of embedding effects), and
aggregate into a contact-like heatmap.

**Cost:** O(N^2) embeddings &mdash; slow but captures non-linear interactions.

In [ ]:
# Generate variant pairs
s = SeqMat.from_fasta('hg38', CHROM, START, END)
df = get_all_epistasis_ids(s.seq, s.index, GENE, CHROM_NUM, STRAND, MAX_DISTANCE)
print(f"{len(df)} epistasis pairs")

# Run RiNALMo
epi_model = EPI_MODEL_CLS()
db = VariantEmbeddingDB(EPI_DB_PATH)

epi_data = add_epistasis_metrics(
    df, db,
    model=epi_model,
    id_col="epistasis_id",
    context=EPI_CONTEXT,
    show_progress=True,
    force=False,
    batch_size=8,
)

# Plot: epi_R_singles
epi_name = EPI_MODEL_CLS.__name__.replace("Wrapper", "")
dep_singles = epi_data.pivot_table(index='pos1', columns='pos2', values='epi_R_singles', aggfunc='max')
dep_singles = dep_singles.combine_first(dep_singles.T)
plot_heatmap(dep_singles.iloc[::-1, ::-1], f"{epi_name}: epi_R_singles (epistasis geometry)")

# Score
r, p = correlate_with_structure(dep_singles.fillna(0).values, M_true, f"{epi_name} epistasis (epi_R_singles)", flip=False)
all_results.append((f"{epi_name} epistasis (epi_R_singles)", r, p))

del epi_model

## 3. SpeciesLM: Log-odds dependency map (MLM head)

Mutate position i, measure how nucleotide probabilities change at position j
via the masked language model head.

**Cost:** O(N) forward passes &mdash; fast, but requires an MLM model.

In [ ]:
depmap_model = DEPMAP_MODEL_CLS()
depmap_name = DEPMAP_MODEL_CLS.__name__.replace("Wrapper", "")
seq_padded = SeqMat.from_fasta('hg38', CHROM, START - PAD, END + PAD).seq
tRNA_positions = list(range(PAD, PAD + END - START + 1))

result_logodds = compute_logodds_dependency_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    show_progress=True,
)
result_logodds.plot(title=f"{depmap_name}: log-odds dependency map")

r, p = correlate_with_structure(result_logodds.matrix, M_true, f"{depmap_name} log-odds")
all_results.append((f"{depmap_name} log-odds", r, p))

## 4. SpeciesLM: Embedding perturbation map (cosine distance)

Mutate position i, re-embed, measure cosine distance at each token position j.

**Cost:** O(N) forward passes &mdash; fast, works with any model (no MLM head needed).

In [ ]:
result_embed = compute_embedding_perturbation_map(
    depmap_model, seq_padded,
    positions=tRNA_positions,
    diff="cosine",
    show_progress=True,
)
result_embed.plot(title=f"{depmap_name}: embedding perturbation map (cosine)")

r, p = correlate_with_structure(result_embed.matrix, M_true, f"{depmap_name} embedding perturbation (cosine)")
all_results.append((f"{depmap_name} embedding perturbation (cosine)", r, p))

del depmap_model

## 5. Summary comparison

In [ ]:
df_results = pd.DataFrame(all_results, columns=['Method', 'Pearson r', 'p-value'])
df_results = df_results.sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_results.index += 1
df_results.index.name = 'Rank'
display(df_results)

# Bar chart
fig, ax = plt.subplots(figsize=(10, max(3, len(df_results) * 0.6)))
colors = ['C2' if 'epistasis' in m else 'C1' if 'log-odds' in m else 'C0' for m in df_results['Method']]
ax.barh(range(len(df_results)), df_results['Pearson r'], color=colors)
ax.set_yticks(range(len(df_results)))
ax.set_yticklabels(df_results['Method'], fontsize=10)
ax.set_xlabel('Pearson r (vs tRNA secondary structure)')
ax.set_title('tRNA41 Contact Prediction: Method Comparison')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='C0', label='Embedding perturbation'),
    Patch(color='C1', label='Log-odds'),
    Patch(color='C2', label='Epistasis geometry'),
], loc='lower right')
plt.tight_layout()
plt.show()

### Interpreting the three methods

The three approaches measure fundamentally different things:

**Embedding perturbation (cosine)** — Mutate position i, re-embed the full sequence,
measure how much the embedding at position j changes (cosine distance). This operates
purely in embedding space with no classification head. It asks: *"does the model's
internal representation of position j care about what's at position i?"*

**Log-odds (KL)** — Mutate position i, use the MLM head to re-predict nucleotide
probabilities at position j. This uses the model's output layer, not the raw embeddings.
It asks: *"does the model's nucleotide prediction at j change when I mutate i?"*

**Epistasis geometry** — Embed all 4 sequences (WT, single-mut-i, single-mut-j,
double-mut-ij), measure non-additivity of the double mutant effect. It asks: *"is the
combined effect of mutating i and j different from the sum of their individual effects?"*

The perturbation and log-odds methods measure **direct dependency** (does j notice i
changed?), while epistasis measures **non-linear interaction** (does mutating both together
produce a surprise?). Two positions can be strongly dependent (high perturbation score)
but perfectly additive (low epistasis) — e.g., two bases in the same stem that both affect
stability independently.

For contact prediction, direct dependency works better than non-additivity. And the
embedding space captures richer information than the MLM head output (cosine > KL),
suggesting the internal representations encode structural relationships that get compressed
when projected down to 4 nucleotide probabilities.

## 6. Beyond contacts: what are the high-scoring non-base-pair positions?

The embedding maps detect more than just Watson-Crick base pairs. Here we identify
position pairs that score highly in the predicted maps but are NOT in the secondary
structure contact map. These may reflect:

- **Stacking interactions** (adjacent bases that stack but don't pair)
- **Tertiary contacts** (D-loop / T-loop interaction, variable loop)
- **Backbone geometry constraints**
- **Covariation signals** from evolutionary pressure

In [ ]:
# Use the best-performing predicted map (embedding perturbation)
# Adjust if a different method scored highest
best_matrix = result_embed.matrix.copy()
best_matrix = best_matrix[::-1, ::-1]
best_matrix[np.isnan(best_matrix)] = 0
np.fill_diagonal(best_matrix, 0)

# tRNA structural regions (standard numbering, 0-indexed into our 74-nt sequence)
# Reverse-complement strand, so positions are mirrored
N = best_matrix.shape[0]
tRNA_regions = {}
for i in range(N):
    if i < 7:
        tRNA_regions[i] = "acceptor stem"
    elif i < 11:
        tRNA_regions[i] = "D-stem"
    elif i < 25:
        tRNA_regions[i] = "D-loop"
    elif i < 32:
        tRNA_regions[i] = "anticodon stem"
    elif i < 38:
        tRNA_regions[i] = "anticodon loop"
    elif i < 44:
        tRNA_regions[i] = "anticodon stem"
    elif i < 48:
        tRNA_regions[i] = "variable loop"
    elif i < 53:
        tRNA_regions[i] = "T-stem"
    elif i < 61:
        tRNA_regions[i] = "T-loop"
    elif i < 66:
        tRNA_regions[i] = "T-stem"
    else:
        tRNA_regions[i] = "acceptor stem"

# Find high-scoring pairs NOT in the contact map
threshold = np.percentile(best_matrix[best_matrix > 0], 90)  # top 10%
novel_pairs = []
i_idx, j_idx = np.where((best_matrix > threshold) & (M_true == 0))
for i, j in zip(i_idx, j_idx):
    if i < j:  # upper triangle only
        novel_pairs.append({
            'pos_i': i, 'pos_j': j,
            'score': best_matrix[i, j],
            'distance': abs(j - i),
            'region_i': tRNA_regions.get(i, '?'),
            'region_j': tRNA_regions.get(j, '?'),
            'adjacent': abs(j - i) == 1,
        })

df_novel = pd.DataFrame(novel_pairs).sort_values('score', ascending=False)
print(f"High-scoring pairs NOT in contact map (top 10%, score > {threshold:.4f}): {len(df_novel)}")
print(f"\nOf these, {df_novel['adjacent'].sum()} are adjacent (likely stacking)")
print(f"\nTop 20 novel interactions:")
display(df_novel.head(20))

# Annotate cross-region contacts (potential tertiary)
cross_region = df_novel[df_novel['region_i'] != df_novel['region_j']]
cross_region_nonadj = cross_region[~cross_region['adjacent']]
print(f"\nCross-region, non-adjacent pairs (potential tertiary contacts): {len(cross_region_nonadj)}")
display(cross_region_nonadj.head(15))

In [ ]:
# Overlay: predicted map with contact map base pairs marked
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(best_matrix, cmap='magma', origin='upper', aspect='equal')

# Mark known base pairs with green circles
bp_i, bp_j = np.where(M_true == 1)
ax.scatter(bp_j, bp_i, s=30, facecolors='none', edgecolors='lime', linewidths=1.5, label='Known base pairs')

# Mark novel high-scoring pairs with cyan X
if len(df_novel) > 0:
    ax.scatter(df_novel['pos_j'], df_novel['pos_i'], s=20, marker='x',
               color='cyan', linewidths=1, label=f'Novel high-scoring (n={len(df_novel)})')

ax.set_xlabel("Position")
ax.set_ylabel("Position")
ax.set_title("Predicted map with known base pairs (green) and novel interactions (cyan)")
ax.legend(loc='lower left', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 7. Compare against 3D distance matrix from PDB

The binary contact map only captures Watson-Crick base pairs. A 3D distance matrix
from a crystal structure captures **all** spatial proximity — stacking, tertiary contacts,
backbone geometry.

We use **PDB 1EHZ** (yeast tRNA-Phe, 1.93 A resolution) as the gold-standard free tRNA
structure. All tRNAs share the same L-shaped 3D fold, so this is a valid structural proxy.
We also try **PDB 1F7U** (yeast tRNA-Arg, 2.2 A) for a same-isotype comparison.

We correlate our predicted maps against `1/distance` (closer = stronger interaction).

In [ ]:
import urllib.request
import gzip
import io

def fetch_pdb(pdb_id):
    """Download a PDB file from RCSB."""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb.gz"
    with urllib.request.urlopen(url) as response:
        return gzip.decompress(response.read()).decode()


def extract_c1prime_coords(pdb_text, chain_id):
    """Extract C1' atom coordinates for each residue in a chain."""
    coords = {}
    for line in pdb_text.splitlines():
        if not line.startswith("ATOM"):
            continue
        atom_name = line[12:16].strip()
        chain = line[21].strip()
        if atom_name != "C1'" or chain != chain_id:
            continue
        resnum = int(line[22:26].strip())
        x = float(line[30:38])
        y = float(line[38:46])
        z = float(line[46:54])
        coords[resnum] = np.array([x, y, z])
    return coords


def coords_to_distance_matrix(coords):
    """Convert residue coords dict to a pairwise distance matrix."""
    resnums = sorted(coords.keys())
    n = len(resnums)
    D = np.zeros((n, n))
    for i, ri in enumerate(resnums):
        for j, rj in enumerate(resnums):
            D[i, j] = np.linalg.norm(coords[ri] - coords[rj])
    return D, resnums


# Fetch structures
PDB_CONFIGS = [
    ('1EHZ', 'A', 'tRNA-Phe (1.93 A, free)'),
    ('1F7U', 'B', 'tRNA-Arg (2.2 A, synthetase complex)'),
]

pdb_distance_matrices = {}
for pdb_id, chain, desc in PDB_CONFIGS:
    print(f"Fetching {pdb_id} chain {chain} ({desc})...")
    pdb_text = fetch_pdb(pdb_id)
    coords = extract_c1prime_coords(pdb_text, chain)
    D, resnums = coords_to_distance_matrix(coords)
    pdb_distance_matrices[pdb_id] = (D, resnums, desc)
    print(f"  {len(resnums)} residues, distance range: {D[D > 0].min():.1f} - {D.max():.1f} A")

In [ ]:
# Compare predicted maps against 3D proximity (1/distance)
# PDB structures may have different lengths than our 74-nt tRNA,
# so we truncate/align to the shorter of the two

predicted_maps = {
    f'{depmap_name} embedding perturbation': result_embed.matrix[::-1, ::-1],
    f'{depmap_name} log-odds': result_logodds.matrix[::-1, ::-1],
    f'{epi_name} epistasis (epi_R_singles)': dep_singles.fillna(0).values,
}

pdb_results = []
for pdb_id, (D, resnums, desc) in pdb_distance_matrices.items():
    n_pdb = D.shape[0]

    # Convert to proximity: 1/distance (zero on diagonal)
    proximity = np.zeros_like(D)
    mask = D > 0
    proximity[mask] = 1.0 / D[mask]
    np.fill_diagonal(proximity, 0)

    # Plot the 3D proximity map
    plt.figure(figsize=(6, 6))
    plt.imshow(proximity, cmap='magma', origin='upper')
    plt.title(f"PDB {pdb_id}: 3D proximity (1/distance)\n{desc}")
    plt.xlabel("Residue")
    plt.ylabel("Residue")
    plt.colorbar(label="1 / distance (A)")
    plt.show()

    for map_name, pred_raw in predicted_maps.items():
        pred = pred_raw.copy()
        pred[np.isnan(pred)] = 0
        np.fill_diagonal(pred, 0)

        # Align sizes: truncate to min(n_pred, n_pdb)
        n_pred = pred.shape[0]
        n = min(n_pred, n_pdb)
        pred_aligned = pred[:n, :n]
        prox_aligned = proximity[:n, :n]

        true_flat = upper_tri_flatten(prox_aligned)
        pred_flat = upper_tri_flatten(pred_aligned)
        r, p = pearsonr(true_flat, pred_flat)
        print(f"  {map_name} vs {pdb_id} ({desc}): Pearson r = {r:.4f}, p = {p:.2e}")
        pdb_results.append({
            'Method': map_name,
            'PDB': f"{pdb_id} ({desc})",
            'Pearson r': r,
            'p-value': p,
        })

print("\n--- Comparison: binary contact map vs 3D proximity as ground truth ---")
df_pdb = pd.DataFrame(pdb_results).sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_pdb.index += 1
df_pdb.index.name = 'Rank'
display(df_pdb)